# Monitoring Rules & Gold Alerts (`pipelines/04_monitoring_rules.ipynb`)

This notebook reads accepted Silver transactions from `workspace.sentinel_silver.transactions`, computes point-in-time historical features using the shared transformation in `pipelines/historical_features.py`, evaluates three versioned monitoring rules with provisional thresholds, and persists flagged alerts to `workspace.sentinel_gold.alerts`.

### Provisional Monitoring Ruleset
| Rule ID | Version | Name | Window (`[lower, upper)`) | Match Condition (Provisional) |
| :--- | :--- | :--- | :--- | :--- |
| `RULE_01_CARD_VELOCITY_1H` | `v1.0-provisional` | Card Velocity (1h) | `[event_time - 1 hour, event_time)` | `card_txn_count_1h >= 3` |
| `RULE_02_UNUSUAL_AMOUNT_14D` | `v1.0-provisional` | Unusual Amount vs. 14d Average | `[event_time - 14 days, event_time)` | `card_txn_count_14d >= 5` and `amount >= 3.00 * card_avg_amount_14d` |
| `RULE_03_REPEATED_SIMILAR_MERCHANT_1H` | `v1.0-provisional` | Repeated Similar Merchant Payments (1h) | `[event_time - 1 hour, event_time)` | `card_merchant_similar_amount_count_1h >= 2` (prior payments to same `(cc_num, merchant)` within `±10%` of current `amount`) |

### Scope & Guardrails
- **Shared feature logic**: Calls `compute_historical_features` from `pipelines/historical_features.py` without copying its implementation.
- **Window semantics**: Strictly earlier timestamps (`< event_time`), inclusive of lower boundaries (`-1 hour`, `-14 days`), excluding equal timestamps and future events.
- **Rule 3 amount similarity**: Uses `card_merchant_similar_amount_count_1h`; `card_merchant_txn_count_1h` alone does not establish similar amounts.
- **Gold persistence (`workspace.sentinel_gold.alerts`)**: One row per flagged transaction with a stable `alert_id`, transaction details, matching rule IDs/versions, and structured `rule_evidence`, rebuilt with `overwrite` mode from real Silver data only (never synthetic test records).
- **Out of scope**: Analyst decisions, workflow status, fraud labels, and additional batch releases.

In [ ]:
# Step 1: Define named provisional rule configuration, load Silver, and verify real-data feature counts
from decimal import Decimal
import os
import sys
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

for _candidate_dir in (os.getcwd(), os.path.join(os.getcwd(), "pipelines")):
    if os.path.isfile(os.path.join(_candidate_dir, "historical_features.py")) and _candidate_dir not in sys.path:
        sys.path.insert(0, _candidate_dir)

from historical_features import (
    FEATURE_COLUMNS,
    assert_feature_grain_and_null_semantics,
    compute_historical_features,
)

SILVER_TRANSACTIONS_TABLE = "workspace.sentinel_silver.transactions"
ACTIVE_RULESET_VERSION = "ruleset-v1.0-provisional"

# Rule 1: Velocity — at least 3 earlier card payments within 1 hour
RULE_01_ID = "RULE_01_CARD_VELOCITY_1H"
RULE_01_VERSION = "v1.0-provisional"
RULE_01_WINDOW = "[event_time - 1h, event_time)"
RULE_01_MIN_PRIOR_CARD_TXNS_1H = 3

# Rule 2: Unusual amount — amount >= 3x earlier 14-day average, with at least 5 earlier payments
RULE_02_ID = "RULE_02_UNUSUAL_AMOUNT_14D"
RULE_02_VERSION = "v1.0-provisional"
RULE_02_WINDOW = "[event_time - 14d, event_time)"
RULE_02_MIN_PRIOR_CARD_TXNS_14D = 5
RULE_02_AMOUNT_MULTIPLIER = Decimal("3.00")

# Rule 3: Repeated similar payments — at least 2 earlier payments to same merchant within 1 hour within +/-10% amount
RULE_03_ID = "RULE_03_REPEATED_SIMILAR_MERCHANT_1H"
RULE_03_VERSION = "v1.0-provisional"
RULE_03_WINDOW = "[event_time - 1h, event_time)"
RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H = 2
RULE_03_AMOUNT_TOLERANCE_PCT = Decimal("0.10")

REQUIRED_SILVER_COLUMNS = {
    "trans_num",
    "cc_num",
    "merchant",
    "category",
    "amount",
    "event_time",
    "replay_batch",
}

silver_transactions = spark.read.table(SILVER_TRANSACTIONS_TABLE)
missing_cols = sorted(REQUIRED_SILVER_COLUMNS - set(silver_transactions.columns))
if missing_cols:
    raise ValueError(f"Missing required Silver columns: {missing_cols}")
if "is_fraud" in silver_transactions.columns:
    raise AssertionError("Label leakage guard failed: 'is_fraud' must not exist in Silver transactions.")

silver_with_features = compute_historical_features(
    silver_transactions,
    similar_amount_tolerance_pct=RULE_03_AMOUNT_TOLERANCE_PCT,
)
real_feature_metrics = assert_feature_grain_and_null_semantics(
    silver_transactions,
    silver_with_features,
)

print(f"Verified real-data historical features from {SILVER_TRANSACTIONS_TABLE}:")
for metric_name, metric_val in real_feature_metrics.items():
    print(f"  {metric_name}: {metric_val:,}")

### Step 2: Rule Evaluation & Supporting Evidence (One Row per Transaction)

`evaluate_monitoring_rules` evaluates all three provisional rules on each transaction row while preserving a strict 1:1 `trans_num` grain:
- **Rule 1 (`RULE_01_CARD_VELOCITY_1H`)**: Matches when `card_txn_count_1h >= 3`.
- **Rule 2 (`RULE_02_UNUSUAL_AMOUNT_14D`)**: Matches when `card_txn_count_14d >= 5`, `card_avg_amount_14d IS NOT NULL`, and `amount >= 3.00 * card_avg_amount_14d`.
- **Rule 3 (`RULE_03_REPEATED_SIMILAR_MERCHANT_1H`)**: Matches when `card_merchant_similar_amount_count_1h >= 2` (prior payments to the same `(cc_num, merchant)` within `[event_time - 1h, event_time)` whose amount is within `[0.90 * amount, 1.10 * amount]`).
- **Supporting Evidence (`rule_evidence`)**: An array of structured records (`rule_id`, `rule_version`, `window`, `observed_metrics`, `threshold_definition`) populated for each matched rule on the transaction.

In [ ]:
# Step 2: Evaluate monitoring rules and display matches and evidence on real Silver data
def evaluate_monitoring_rules(feature_df: DataFrame) -> DataFrame:
    """
    Evaluate the three provisional monitoring rules per transaction (1 row per trans_num).
    Preserves all matching rule IDs, rule versions, and structured supporting evidence.
    """
    lower_factor = F.lit(Decimal("1.00") - RULE_03_AMOUNT_TOLERANCE_PCT)
    upper_factor = F.lit(Decimal("1.00") + RULE_03_AMOUNT_TOLERANCE_PCT)

    with_bounds = (
        feature_df.withColumn(
            "unusual_amount_threshold_14d",
            F.when(
                F.col("card_avg_amount_14d").isNotNull(),
                F.round(
                    F.col("card_avg_amount_14d") * F.lit(RULE_02_AMOUNT_MULTIPLIER), 2
                ).cast("decimal(18,2)"),
            ).otherwise(F.lit(None).cast("decimal(18,2)")),
        )
        .withColumn(
            "similar_amount_lower_bound",
            F.round(F.col("amount") * lower_factor, 2).cast("decimal(18,2)"),
        )
        .withColumn(
            "similar_amount_upper_bound",
            F.round(F.col("amount") * upper_factor, 2).cast("decimal(18,2)"),
        )
    )

    rule_01_cond = F.col("card_txn_count_1h") >= F.lit(RULE_01_MIN_PRIOR_CARD_TXNS_1H)
    rule_02_cond = (
        (F.col("card_txn_count_14d") >= F.lit(RULE_02_MIN_PRIOR_CARD_TXNS_14D))
        & F.col("card_avg_amount_14d").isNotNull()
        & (
            F.col("amount")
            >= (F.col("card_avg_amount_14d") * F.lit(RULE_02_AMOUNT_MULTIPLIER))
        )
    )
    rule_03_cond = F.col("card_merchant_similar_amount_count_1h") >= F.lit(
        RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H
    )

    with_flags = (
        with_bounds.withColumn("rule_01_velocity_match", rule_01_cond)
        .withColumn("rule_02_unusual_amount_match", rule_02_cond)
        .withColumn("rule_03_similar_merchant_match", rule_03_cond)
    )

    rule_01_evidence = F.when(
        F.col("rule_01_velocity_match"),
        F.struct(
            F.lit(RULE_01_ID).alias("rule_id"),
            F.lit(RULE_01_VERSION).alias("rule_version"),
            F.lit(RULE_01_WINDOW).alias("window"),
            F.concat(
                F.lit("card_txn_count_1h="),
                F.col("card_txn_count_1h").cast("string"),
            ).alias("observed_metrics"),
            F.lit(
                f"card_txn_count_1h >= {RULE_01_MIN_PRIOR_CARD_TXNS_1H}"
            ).alias("threshold_definition"),
        ),
    )

    rule_02_evidence = F.when(
        F.col("rule_02_unusual_amount_match"),
        F.struct(
            F.lit(RULE_02_ID).alias("rule_id"),
            F.lit(RULE_02_VERSION).alias("rule_version"),
            F.lit(RULE_02_WINDOW).alias("window"),
            F.concat(
                F.lit("amount="),
                F.col("amount").cast("string"),
                F.lit(", card_avg_amount_14d="),
                F.col("card_avg_amount_14d").cast("string"),
                F.lit(", card_txn_count_14d="),
                F.col("card_txn_count_14d").cast("string"),
            ).alias("observed_metrics"),
            F.concat(
                F.lit(
                    f"card_txn_count_14d >= {RULE_02_MIN_PRIOR_CARD_TXNS_14D} AND amount >= {RULE_02_AMOUNT_MULTIPLIER} * card_avg_amount_14d ("
                ),
                F.col("unusual_amount_threshold_14d").cast("string"),
                F.lit(")"),
            ).alias("threshold_definition"),
        ),
    )

    rule_03_evidence = F.when(
        F.col("rule_03_similar_merchant_match"),
        F.struct(
            F.lit(RULE_03_ID).alias("rule_id"),
            F.lit(RULE_03_VERSION).alias("rule_version"),
            F.lit(RULE_03_WINDOW).alias("window"),
            F.concat(
                F.lit("amount="),
                F.col("amount").cast("string"),
                F.lit(", card_merchant_similar_amount_count_1h="),
                F.col("card_merchant_similar_amount_count_1h").cast("string"),
                F.lit(", card_merchant_txn_count_1h="),
                F.col("card_merchant_txn_count_1h").cast("string"),
            ).alias("observed_metrics"),
            F.concat(
                F.lit(
                    f"card_merchant_similar_amount_count_1h >= {RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H} within +/-10% ["
                ),
                F.col("similar_amount_lower_bound").cast("string"),
                F.lit(", "),
                F.col("similar_amount_upper_bound").cast("string"),
                F.lit("]"),
            ).alias("threshold_definition"),
        ),
    )

    raw_evidence_array = F.array(rule_01_evidence, rule_02_evidence, rule_03_evidence)
    filtered_evidence = F.filter(raw_evidence_array, lambda item: item.isNotNull())

    return (
        with_flags.withColumn("ruleset_version", F.lit(ACTIVE_RULESET_VERSION))
        .withColumn("rule_evidence", filtered_evidence)
        .withColumn(
            "matched_rule_ids",
            F.transform(F.col("rule_evidence"), lambda item: item["rule_id"]),
        )
        .withColumn(
            "matched_rule_versions",
            F.transform(F.col("rule_evidence"), lambda item: item["rule_version"]),
        )
        .withColumn("matched_rule_count", F.size(F.col("matched_rule_ids")))
        .withColumn("is_flagged", F.col("matched_rule_count") > 0)
    )


def assert_rule_evaluation_grain(
    input_df: DataFrame, evaluated_df: DataFrame
) -> dict[str, int]:
    """Verify 1:1 transaction grain and rule-match consistency."""
    summary = (
        evaluated_df.agg(
            F.count("*").alias("evaluated_rows"),
            F.countDistinct("trans_num").alias("distinct_trans_num"),
            F.sum(F.when(F.col("is_flagged"), 1).otherwise(0)).alias("flagged_transactions"),
            F.sum(F.when(F.col("rule_01_velocity_match"), 1).otherwise(0)).alias("rule_01_matches"),
            F.sum(F.when(F.col("rule_02_unusual_amount_match"), 1).otherwise(0)).alias("rule_02_matches"),
            F.sum(F.when(F.col("rule_03_similar_merchant_match"), 1).otherwise(0)).alias("rule_03_matches"),
            F.sum(F.when(F.col("matched_rule_count") > 1, 1).otherwise(0)).alias("multi_rule_transactions"),
        )
        .collect()[0]
        .asDict()
    )

    input_rows = input_df.count()
    if summary["evaluated_rows"] != input_rows:
        raise AssertionError(
            f"Evaluated row count ({summary['evaluated_rows']}) != input row count ({input_rows})"
        )
    if summary["distinct_trans_num"] != input_rows:
        raise AssertionError(
            f"Duplicate trans_num in evaluated output: {summary['evaluated_rows']} vs {summary['distinct_trans_num']}"
        )

    return {k: int(v or 0) for k, v in summary.items()}


real_evaluated_df = evaluate_monitoring_rules(silver_with_features)
real_rule_summary = assert_rule_evaluation_grain(silver_transactions, real_evaluated_df)

print("Real-data monitoring rule evaluation summary:")
for k, v in real_rule_summary.items():
    print(f"  {k}: {v:,}")

print("\nFlagged transactions with supporting evidence (up to 25 rows):")
display(
    real_evaluated_df.filter(F.col("is_flagged"))
    .select(
        "trans_num",
        "cc_num",
        "merchant",
        "amount",
        "event_time",
        *FEATURE_COLUMNS,
        "unusual_amount_threshold_14d",
        "matched_rule_ids",
        "matched_rule_versions",
        "rule_evidence",
    )
    .orderBy("event_time", "trans_num")
    .limit(25)
)

### Step 3: Synthetic Verification Suite (Matches, Non-Matches, Exact Boundaries & Insufficient History)

The synthetic suite below exercises `compute_historical_features` + `evaluate_monitoring_rules` directly to verify:
1. **Rule 1 (`RULE_01_CARD_VELOCITY_1H`)**:
   - Insufficient history / non-match at `0`, `1`, and `2` prior card payments in `1h`.
   - Exact count and lower time boundary match when the 3rd prior payment is at `event_time - 1 hour` (`-3,600s`, `card_txn_count_1h == 3`).
   - Lower time boundary exclusion when a prior payment falls `1 second` outside the 1-hour window (`-3,601s`, `card_txn_count_1h == 2`).
   - Equal-timestamp exclusion when two transactions occur at the same timestamp with `2` prior payments (`card_txn_count_1h == 2` on both).
2. **Rule 2 (`RULE_02_UNUSUAL_AMOUNT_14D`)**:
   - No history (`0` prior payments, `card_avg_amount_14d IS NULL`) does not match.
   - Insufficient history (`4` prior payments `< 5`, even with `amount >= 3x` average) does not match.
   - Just below `3x` threshold (`5` prior payments with average `20.00`, `amount = 59.99 < 60.00`) does not match.
   - Exact count (`5` prior payments, including one at `event_time - 14 days`) and exact amount boundary (`amount = 60.00 == 3.00 * 20.00`) matches.
   - Lower 14-day time boundary exclusion when the earliest of 5 prior payments falls at `-14 days - 1 second` (`card_txn_count_14d == 4 < 5`).
3. **Rule 3 (`RULE_03_REPEATED_SIMILAR_MERCHANT_1H`)**:
   - Merchant count alone does not match: `2` prior payments to the same merchant in `1h` with dissimilar amounts (`50.00` and `110.01` vs. current `100.00` -> `card_merchant_txn_count_1h == 2`, `card_merchant_similar_amount_count_1h == 0`).
   - Exact `-10%` (`90.00` at `event_time - 1 hour`) and `+10%` (`110.00` at `-30m`) boundaries vs. current `100.00` (`card_merchant_similar_amount_count_1h == 2`) matches Rule 3 alone (`[RULE_03_ID]`).
   - Lower 1-hour boundary exclusion (`-1h - 1s`, `10:59:59` vs. `12:00:00`) leaves only `1` similar payment (`< 2`) and does not match.
   - Multi-rule transaction: a subsequent payment that satisfies both Rule 1 (`>= 3` card payments in `1h`) and Rule 3 (`>= 2` similar merchant payments in `1h`) produces a single row with both rule IDs (`[RULE_01_ID, RULE_03_ID]`) and evidence structs.

In [ ]:
# Step 3: Execute synthetic tests for matches, non-matches, exact boundaries, and insufficient history
synthetic_rows = [
    # --- Card R1: Velocity (Rule 1) boundary & equal-timestamp tests ---
    # Anchor at 2019-01-15 12:00:00:
    # r1_tx1 at 11:00:00 (exact -1h from 12:00:00), r1_tx2 at 11:20:00, r1_tx3 at 11:40:00
    # Use distinct merchants and decreasing amounts so Rules 2 and 3 do not fire.
    ("r1_tx1_no_hist",       "card_R1", "M_A", "misc", Decimal("100.00"), "2019-01-15 11:00:00"),
    ("r1_tx2_1_prior",       "card_R1", "M_B", "misc", Decimal("90.00"),  "2019-01-15 11:20:00"),
    ("r1_tx3_2_prior",       "card_R1", "M_C", "misc", Decimal("80.00"),  "2019-01-15 11:40:00"),
    # At 12:00:00 (exact +1h from r1_tx1): sees r1_tx1, r1_tx2, r1_tx3 -> card_txn_count_1h == 3 -> Rule 1 MATCH
    ("r1_tx4_exact_1h_3",    "card_R1", "M_D", "misc", Decimal("70.00"),  "2019-01-15 12:00:00"),
    # At 12:20:01 (r1_tx1 & r1_tx2 are >1h ago; sees only r1_tx3 & r1_tx4 -> card_txn_count_1h == 2);
    # two equal-timestamp transactions must not count each other -> both stay at 2 -> Rule 1 NON-MATCH
    ("r1_tx5_eq_ts_a",       "card_R1", "M_E", "misc", Decimal("60.00"),  "2019-01-15 12:20:01"),
    ("r1_tx6_eq_ts_b",       "card_R1", "M_F", "misc", Decimal("50.00"),  "2019-01-15 12:20:01"),

    # --- Card R2: Unusual Amount (Rule 2) history count, 3x boundary & 14d window tests ---
    # Anchor at 2019-01-15 12:00:00; exact -14d is 2019-01-01 12:00:00.
    # Spacing > 1h apart so Rule 1 and Rule 3 never fire.
    ("r2_tx1_exact_14d",     "card_R2", "M_A", "misc", Decimal("20.00"),  "2019-01-01 12:00:00"),
    ("r2_tx2_hist_2",        "card_R2", "M_B", "misc", Decimal("20.00"),  "2019-01-03 12:00:00"),
    ("r2_tx3_hist_3",        "card_R2", "M_C", "misc", Decimal("20.00"),  "2019-01-05 12:00:00"),
    ("r2_tx4_hist_4",        "card_R2", "M_D", "misc", Decimal("20.00"),  "2019-01-07 12:00:00"),
    # 5th transaction has only 4 prior txns (avg=20.00): even with amount=100.00 (>=3x), count=4 < 5 -> NON-MATCH
    ("r2_tx5_only_4_prior",  "card_R2", "M_E", "misc", Decimal("100.00"), "2019-01-09 12:00:00"),

    # Separate card_R2B with 5 prior txns of 20.00 each (avg = 20.00, 3x threshold = 60.00)
    ("r2b_tx1_exact_14d",    "card_R2B", "M_A", "misc", Decimal("20.00"), "2019-01-01 12:00:00"),
    ("r2b_tx2_hist_2",       "card_R2B", "M_B", "misc", Decimal("20.00"), "2019-01-03 12:00:00"),
    ("r2b_tx3_hist_3",       "card_R2B", "M_C", "misc", Decimal("20.00"), "2019-01-05 12:00:00"),
    ("r2b_tx4_hist_4",       "card_R2B", "M_D", "misc", Decimal("20.00"), "2019-01-07 12:00:00"),
    ("r2b_tx5_hist_5",       "card_R2B", "M_E", "misc", Decimal("20.00"), "2019-01-09 12:00:00"),
    # At 2019-01-14 12:00:00: 5 prior txns (avg=20.00), amount=59.99 (1 cent below 3x=60.00) -> NON-MATCH
    ("r2b_tx6_below_3x",     "card_R2B", "M_F", "misc", Decimal("59.99"), "2019-01-14 12:00:00"),

    # Separate card_R2C to test exact 3x boundary at exact +14d from tx1
    ("r2c_tx1_exact_14d",    "card_R2C", "M_A", "misc", Decimal("20.00"), "2019-01-01 12:00:00"),
    ("r2c_tx2_hist_2",       "card_R2C", "M_B", "misc", Decimal("20.00"), "2019-01-03 12:00:00"),
    ("r2c_tx3_hist_3",       "card_R2C", "M_C", "misc", Decimal("20.00"), "2019-01-05 12:00:00"),
    ("r2c_tx4_hist_4",       "card_R2C", "M_D", "misc", Decimal("20.00"), "2019-01-07 12:00:00"),
    ("r2c_tx5_hist_5",       "card_R2C", "M_E", "misc", Decimal("20.00"), "2019-01-09 12:00:00"),
    # At 2019-01-15 12:00:00 (r2c_tx1 is at exact -14d): 5 prior txns, avg=20.00, amount=60.00 (exact 3x) -> Rule 2 MATCH
    ("r2c_tx6_exact_3x",     "card_R2C", "M_F", "misc", Decimal("60.00"), "2019-01-15 12:00:00"),

    # Card R2D: earliest of 5 prior txns is at -14d - 1s (2019-01-01 11:59:59) so at 2019-01-15 12:00:00 only 4 remain -> NON-MATCH
    ("r2d_tx1_before_14d",   "card_R2D", "M_A", "misc", Decimal("20.00"), "2019-01-01 11:59:59"),
    ("r2d_tx2_hist_2",       "card_R2D", "M_B", "misc", Decimal("20.00"), "2019-01-03 12:00:00"),
    ("r2d_tx3_hist_3",       "card_R2D", "M_C", "misc", Decimal("20.00"), "2019-01-05 12:00:00"),
    ("r2d_tx4_hist_4",       "card_R2D", "M_D", "misc", Decimal("20.00"), "2019-01-07 12:00:00"),
    ("r2d_tx5_hist_5",       "card_R2D", "M_E", "misc", Decimal("20.00"), "2019-01-09 12:00:00"),
    ("r2d_tx6_dropped_14d",  "card_R2D", "M_F", "misc", Decimal("60.00"), "2019-01-15 12:00:00"),

    # --- Card R3A: Merchant count >= 2 with dissimilar amounts does NOT match Rule 3 ---
    ("r3a_tx1_dissim_low",   "card_R3A", "M_SIM", "retail", Decimal("50.00"),  "2019-01-15 11:10:00"),
    ("r3a_tx2_dissim_high",  "card_R3A", "M_SIM", "retail", Decimal("110.01"), "2019-01-15 11:20:00"),
    # At 11:30:00 with amount=100.00: merchant count is 2, but 50.00 (<90.00) and 110.01 (>110.00) are outside +/-10% -> similar count=0 -> NON-MATCH
    ("r3a_tx3_merch2_sim0",  "card_R3A", "M_SIM", "retail", Decimal("100.00"), "2019-01-15 11:30:00"),

    # --- Card R3B: Exact +/-10% amount boundaries at exact -1h (Rule 3 only), then multi-rule (Rule 1 + Rule 3) ---
    ("r3b_tx1_exact_m10pct", "card_R3B", "M_SIM", "retail", Decimal("90.00"),  "2019-01-15 11:00:00"),
    ("r3b_tx2_exact_p10pct", "card_R3B", "M_SIM", "retail", Decimal("110.00"), "2019-01-15 11:30:00"),
    # At 12:00:00 (exact +1h from r3b_tx1): amount=100.00 sees 90.00 (exact -10%) and 110.00 (exact +10%)
    # -> card_txn_count_1h=2 (<3, Rule 1 False), card_merchant_similar_amount_count_1h=2 (>=2, Rule 3 True)
    ("r3b_tx3_exact_sim2",   "card_R3B", "M_SIM", "retail", Decimal("100.00"), "2019-01-15 12:00:00"),
    # Add one payment at another merchant M_OTHER at 12:05:00; then at 12:10:00 (r3b_tx1 is >1h ago),
    # prior 1h has r3b_tx2 (110.00, M_SIM), r3b_tx3 (100.00, M_SIM), and r3b_tx3b (25.00, M_OTHER)
    # -> card_txn_count_1h=3 AND card_merchant_similar_amount_count_1h=2 -> BOTH Rule 1 & Rule 3 MATCH!
    ("r3b_tx3b_other_merch", "card_R3B", "M_OTHER", "gas",  Decimal("25.00"),  "2019-01-15 12:05:00"),
    ("r3b_tx4_multi_r1_r3",  "card_R3B", "M_SIM", "retail", Decimal("100.00"), "2019-01-15 12:10:00"),

    # --- Card R3C: -1h - 1s lower boundary exclusion for Rule 3 ---
    ("r3c_tx1_before_1h",    "card_R3C", "M_SIM", "retail", Decimal("90.00"),  "2019-01-15 10:59:59"),
    ("r3c_tx2_in_1h",        "card_R3C", "M_SIM", "retail", Decimal("110.00"), "2019-01-15 11:30:00"),
    # At 12:00:00, r3c_tx1 is -1h - 1s ago (excluded), so only 1 similar payment remains (<2) -> NON-MATCH
    ("r3c_tx3_dropped_1h",   "card_R3C", "M_SIM", "retail", Decimal("100.00"), "2019-01-15 12:00:00"),
]

synthetic_input_df = (
    spark.createDataFrame(
        synthetic_rows,
        schema=["trans_num", "cc_num", "merchant", "category", "amount", "event_time_str"],
    )
    .withColumn("event_time", F.to_timestamp_ntz("event_time_str"))
    .drop("event_time_str")
)

synthetic_features_df = compute_historical_features(
    synthetic_input_df,
    similar_amount_tolerance_pct=RULE_03_AMOUNT_TOLERANCE_PCT,
)
assert_feature_grain_and_null_semantics(synthetic_input_df, synthetic_features_df)

synthetic_evaluated_df = evaluate_monitoring_rules(synthetic_features_df)
synthetic_summary = assert_rule_evaluation_grain(synthetic_input_df, synthetic_evaluated_df)

actual_outcomes = {
    row["trans_num"]: (
        row["card_txn_count_1h"],
        row["card_txn_count_14d"],
        row["card_avg_amount_14d"],
        row["card_merchant_txn_count_1h"],
        row["card_merchant_similar_amount_count_1h"],
        row["matched_rule_ids"],
    )
    for row in synthetic_evaluated_df.collect()
}

expected_outcomes = {
    # Rule 1 checks
    "r1_tx1_no_hist":       (0, 0, None,              0, 0, []),
    "r1_tx2_1_prior":       (1, 1, Decimal("100.00"), 0, 0, []),
    "r1_tx3_2_prior":       (2, 2, Decimal("95.00"),  0, 0, []),
    "r1_tx4_exact_1h_3":    (3, 3, Decimal("90.00"),  0, 0, [RULE_01_ID]),
    "r1_tx5_eq_ts_a":       (2, 4, Decimal("85.00"),  0, 0, []),
    "r1_tx6_eq_ts_b":       (2, 4, Decimal("85.00"),  0, 0, []),
    # Rule 2 checks
    "r2_tx5_only_4_prior":  (0, 4, Decimal("20.00"),  0, 0, []),
    "r2b_tx6_below_3x":     (0, 5, Decimal("20.00"),  0, 0, []),
    "r2c_tx6_exact_3x":     (0, 5, Decimal("20.00"),  0, 0, [RULE_02_ID]),
    "r2d_tx6_dropped_14d":  (0, 4, Decimal("20.00"),  0, 0, []),
    # Rule 3 checks (dissimilar merchant amounts, exact +/-10% at -1h, multi-rule, and -1h-1s exclusion)
    "r3a_tx3_merch2_sim0":  (2, 2, Decimal("80.01"),  2, 0, []),
    "r3b_tx1_exact_m10pct": (0, 0, None,              0, 0, []),
    "r3b_tx2_exact_p10pct": (1, 1, Decimal("90.00"),  1, 0, []),
    "r3b_tx3_exact_sim2":   (2, 2, Decimal("100.00"), 2, 2, [RULE_03_ID]),
    "r3b_tx3b_other_merch": (2, 3, Decimal("100.00"), 0, 0, []),
    "r3b_tx4_multi_r1_r3":  (3, 4, Decimal("81.25"),  2, 2, [RULE_01_ID, RULE_03_ID]),
    "r3c_tx3_dropped_1h":   (1, 2, Decimal("100.00"), 1, 1, []),
}

for tx_id, expected_tuple in expected_outcomes.items():
    actual_tuple = actual_outcomes.get(tx_id)
    if actual_tuple != expected_tuple:
        raise AssertionError(
            f"Synthetic rule test failed for {tx_id}: expected {expected_tuple}, got {actual_tuple}"
        )

print("All synthetic rule tests passed (matches, non-matches, exact boundaries, insufficient history, multi-rule grain).")
print("Synthetic rule summary:", synthetic_summary)
display(
    synthetic_evaluated_df.select(
        "trans_num",
        "cc_num",
        "merchant",
        "amount",
        "event_time",
        *FEATURE_COLUMNS,
        "unusual_amount_threshold_14d",
        "matched_rule_ids",
        "rule_evidence",
    ).orderBy("cc_num", "event_time", "trans_num")
)

### Step 4: Gold Alerts Persistence (`workspace.sentinel_gold.alerts`)

For this prototype, we persist flagged transactions from the real Silver table (`workspace.sentinel_silver.transactions`) into `workspace.sentinel_gold.alerts` using full-snapshot `overwrite` mode:
- **One row per flagged transaction (`trans_num`)**: Unflagged transactions are excluded from `workspace.sentinel_gold.alerts`.
- **Stable alert identifier (`alert_id`)**: Computed deterministically as `SHA-256(ruleset_version || '|' || trans_num)`, ensuring identical `alert_id` values across equivalent reruns.
- **Synthetic fixture isolation**: Reads directly from `workspace.sentinel_silver.transactions` and asserts zero overlap with `synthetic_rows` before writing.
- **Explicit curated projection (`GOLD_ALERTS_COLUMNS`)**: Retains transaction details, observed historical features, matching rule IDs/versions, and structured `rule_evidence`, while excluding fraud labels and out-of-scope analyst workflow fields.

In [ ]:
# Step 4: Build and persist Gold alerts from real Silver transactions (overwrite mode)
GOLD_SCHEMA = "workspace.sentinel_gold"
GOLD_ALERTS_TABLE = f"{GOLD_SCHEMA}.alerts"

GOLD_ALERTS_COLUMNS = [
    "alert_id",
    "trans_num",
    "ruleset_version",
    "event_time",
    "cc_num",
    "merchant",
    "category",
    "amount",
    "replay_batch",
    "source_file",
    "source_row_index",
    "card_txn_count_1h",
    "card_txn_count_14d",
    "card_avg_amount_14d",
    "unusual_amount_threshold_14d",
    "card_merchant_txn_count_1h",
    "card_merchant_similar_amount_count_1h",
    "similar_amount_lower_bound",
    "similar_amount_upper_bound",
    "matched_rule_count",
    "matched_rule_ids",
    "matched_rule_versions",
    "rule_evidence",
]

SYNTHETIC_RULE_TEST_IDS = [row[0] for row in synthetic_rows]


def build_real_gold_alerts() -> tuple[DataFrame, dict[str, int]]:
    """
    Recompute features and monitoring rules from real Silver transactions,
    verify zero synthetic test rows are present, and project Gold alert rows.
    """
    silver_for_gold = spark.read.table(SILVER_TRANSACTIONS_TABLE)
    if "is_fraud" in silver_for_gold.columns:
        raise AssertionError("Label leakage guard failed: 'is_fraud' present in Silver input.")

    synthetic_leak_count = (
        silver_for_gold.filter(F.col("trans_num").isin(SYNTHETIC_RULE_TEST_IDS)).count()
    )
    if synthetic_leak_count != 0:
        raise AssertionError(
            f"Refusing to persist Gold alerts: found {synthetic_leak_count} synthetic test rows in Silver."
        )

    features_for_gold = compute_historical_features(
        silver_for_gold,
        similar_amount_tolerance_pct=RULE_03_AMOUNT_TOLERANCE_PCT,
    )
    assert_feature_grain_and_null_semantics(silver_for_gold, features_for_gold)

    evaluated_for_gold = evaluate_monitoring_rules(features_for_gold)
    eval_summary = assert_rule_evaluation_grain(silver_for_gold, evaluated_for_gold)

    gold_alerts_df = (
        evaluated_for_gold.filter(F.col("is_flagged"))
        .withColumn(
            "alert_id",
            F.sha2(F.concat_ws("|", F.col("ruleset_version"), F.col("trans_num")), 256),
        )
        .select(*GOLD_ALERTS_COLUMNS)
    )
    return gold_alerts_df, eval_summary


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

expected_gold_alerts_df, gold_eval_summary = build_real_gold_alerts()
expected_gold_alerts_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(GOLD_ALERTS_TABLE)

print(f"Persisted {GOLD_ALERTS_TABLE} in full-snapshot overwrite mode.")
print(f"Expected flagged alerts written: {gold_eval_summary['flagged_transactions']:,}")

### Step 5: Post-Write Read-Back Verification & Rerun Idempotency Check

We read `workspace.sentinel_gold.alerts` back from Delta storage to verify:
1. **Row counts & key uniqueness**: Persisted row count matches `flagged_transactions`, with 100% unique `alert_id` and `trans_num` values and zero synthetic test IDs.
2. **Retained rule metadata & evidence**: Every persisted alert has `matched_rule_count >= 1`, matching array lengths for `matched_rule_ids`, `matched_rule_versions`, and `rule_evidence`, and non-null evidence fields.
3. **Multiplicity-preserving content equality**: `exceptAll` in both directions against `expected_gold_alerts_df` is `0`.
4. **Rerun check**: Rebuilding and overwriting `workspace.sentinel_gold.alerts` a second time produces the exact same row count, stable `alert_id` values, and identical table contents (`exceptAll == 0`).

In [ ]:
# Step 5: Read back workspace.sentinel_gold.alerts, verify counts/IDs/evidence, and run rerun check
def verify_persisted_gold_alerts(
    expected_df: DataFrame, expected_flagged_count: int
) -> DataFrame:
    """Read back workspace.sentinel_gold.alerts and verify schema, counts, unique IDs, and evidence."""
    saved_df = spark.read.table(GOLD_ALERTS_TABLE)

    if saved_df.columns != GOLD_ALERTS_COLUMNS:
        raise AssertionError(
            f"Gold schema mismatch: expected {GOLD_ALERTS_COLUMNS}, got {saved_df.columns}"
        )
    if "is_fraud" in saved_df.columns:
        raise AssertionError("Label leakage guard failed: 'is_fraud' present in Gold alerts.")

    checks = (
        saved_df.agg(
            F.count("*").alias("saved_rows"),
            F.countDistinct("alert_id").alias("distinct_alert_ids"),
            F.countDistinct("trans_num").alias("distinct_trans_nums"),
            F.sum(
                F.when(F.col("trans_num").isin(SYNTHETIC_RULE_TEST_IDS), 1).otherwise(0)
            ).alias("synthetic_rows_in_gold"),
            F.sum(
                F.when(
                    F.col("alert_id").isNull()
                    | F.col("trans_num").isNull()
                    | (F.col("matched_rule_count") < 1)
                    | (F.size(F.col("matched_rule_ids")) != F.col("matched_rule_count"))
                    | (F.size(F.col("matched_rule_versions")) != F.col("matched_rule_count"))
                    | (F.size(F.col("rule_evidence")) != F.col("matched_rule_count"))
                    | F.exists(
                        F.col("rule_evidence"),
                        lambda ev: ev["rule_id"].isNull()
                        | ev["rule_version"].isNull()
                        | ev["observed_metrics"].isNull()
                        | ev["threshold_definition"].isNull(),
                    ),
                    1,
                ).otherwise(0)
            ).alias("invalid_evidence_or_id_rows"),
        )
        .collect()[0]
        .asDict()
    )

    if checks["saved_rows"] != expected_flagged_count:
        raise AssertionError(
            f"Saved Gold alert count ({checks['saved_rows']}) != expected ({expected_flagged_count})"
        )
    if checks["distinct_alert_ids"] != expected_flagged_count:
        raise AssertionError(
            f"Non-unique alert_id in Gold alerts: {checks['distinct_alert_ids']} vs {checks['saved_rows']}"
        )
    if checks["distinct_trans_nums"] != expected_flagged_count:
        raise AssertionError(
            f"Non-unique trans_num in Gold alerts: {checks['distinct_trans_nums']} vs {checks['saved_rows']}"
        )
    if int(checks["synthetic_rows_in_gold"] or 0) != 0:
        raise AssertionError("Synthetic test rows detected in workspace.sentinel_gold.alerts!")
    if int(checks["invalid_evidence_or_id_rows"] or 0) != 0:
        raise AssertionError(
            f"Found {checks['invalid_evidence_or_id_rows']} Gold rows with missing IDs or incomplete rule_evidence."
        )

    diff_expected_vs_saved = expected_df.exceptAll(saved_df).count()
    diff_saved_vs_expected = saved_df.exceptAll(expected_df).count()
    if diff_expected_vs_saved != 0 or diff_saved_vs_expected != 0:
        raise AssertionError(
            f"Gold content mismatch: expected_minus_saved={diff_expected_vs_saved}, saved_minus_expected={diff_saved_vs_expected}"
        )

    return saved_df


# First read-back verification
first_saved_gold_df = verify_persisted_gold_alerts(
    expected_gold_alerts_df,
    gold_eval_summary["flagged_transactions"],
)
first_run_snapshot = spark.createDataFrame(
    first_saved_gold_df.collect(),
    schema=first_saved_gold_df.schema,
)

# Rerun check: rebuild and overwrite workspace.sentinel_gold.alerts a second time
rerun_expected_gold_df, rerun_eval_summary = build_real_gold_alerts()
rerun_expected_gold_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(GOLD_ALERTS_TABLE)

second_saved_gold_df = verify_persisted_gold_alerts(
    rerun_expected_gold_df,
    rerun_eval_summary["flagged_transactions"],
)

rerun_diff_forward = first_run_snapshot.exceptAll(second_saved_gold_df).count()
rerun_diff_reverse = second_saved_gold_df.exceptAll(first_run_snapshot).count()
if rerun_diff_forward != 0 or rerun_diff_reverse != 0:
    raise AssertionError(
        f"Gold rerun idempotency failed: forward_diff={rerun_diff_forward}, reverse_diff={rerun_diff_reverse}"
    )

print(
    f"Verified {GOLD_ALERTS_TABLE}: {second_saved_gold_df.count():,} alerts "
    "(unique alert_id, unique trans_num, retained rule_evidence, and identical rerun output)."
)
display(second_saved_gold_df.orderBy("event_time", "trans_num"))